# 02 — Simulação MZA-FAO do milho 2ª safra (399 municípios × 27 anos × 9 datas)

Roda `modelo_mza.simular` (balanço hídrico de cultura, PPf e PA da biblioteca
**agrometeorologiapy 0.3.0**) para cada município do Paraná, ano de semeadura 1999–2025 e nove datas
de semeadura (dias 5, 15 e 25 de janeiro a março), com o clima diário BR-DWGD do notebook 01.

Escolhas do modelo (ver `modelo_mza.py`):

- **Fenologia por graus-dia** (Tb = 10 °C, 1450 °C dia até a maturação fisiológica): a semeadura
  tardia alonga o ciclo e desloca o enchimento de grãos para o inverno.
- **Kc** nas quatro fases FAO-56 (0,3 / 1,2 / 0,6) aplicado à ETo Penman-Monteith do BR-DWGD.
- **ky** nos cinco períodos de Doorenbos & Kassam (1979): estabelecimento 0,4; vegetativo 0,4;
  floração 1,5; enchimento 0,5; maturação 0,2 — PA pelo produtório por fase.
- **Raiz** de 0,10 m a 0,40 m até o fim do vegetativo; CAD = z × DTA municipal.
- 30 dias de balanço antes da semeadura (palhada, Kc inicial) para o armazenamento inicial.
- **Critérios do ZARC** (Portaria SPA/MAPA nº 329/2026) num ciclo de referência de 100 dias, com as
  fases em dias fixos da portaria, para a classificação oficial de risco.

Leva ~25 min em 4 núcleos.

In [1]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import pandas as pd

import modelo_mza as M
from sensibilidade import PARAM, SAIDA

RAIZ = Path.cwd().parent
DADOS, RESULTADOS = RAIZ / "dados", RAIZ / "resultados"

ANOS = range(1999, 2026)
DATAS_SEMEADURA = [(m, d) for m in (1, 2, 3) for d in (5, 15, 25)]
CT = PARAM.get("ct", M.CT)  # soma térmica (°C dia); muda só nos cenários de sensibilidade

municipios = pd.read_csv(DADOS / "municipios_pr.csv")
print(len(municipios), "municípios ×", len(ANOS), "anos ×", len(DATAS_SEMEADURA), "datas =",
      len(municipios) * len(ANOS) * len(DATAS_SEMEADURA), "simulações")

399 municípios × 27 anos × 9 datas = 96957 simulações


In [2]:
# leitura por município: parquet particionável evita carregar 4 milhões de linhas em cada processo
if not (DADOS / "xavier_diario_pr.parquet").exists():
    pd.read_csv(DADOS / "xavier_diario_pr.csv.gz", parse_dates=["data"]).to_parquet(
        DADOS / "xavier_diario_pr.parquet", index=False)

arquivo = DADOS / "xavier_diario_pr.parquet"
with ProcessPoolExecutor(max_workers=4) as ex:
    futuros = [ex.submit(M.simular_municipio, r.cod_ibge, r.latitude, r.dta_mm_m, arquivo, ANOS, DATAS_SEMEADURA, CT)
               for r in municipios.itertuples()]
    sim = pd.concat([f.result() for f in futuros], ignore_index=True)

sim.to_parquet(SAIDA / "simulacao_mza_pr.parquet", index=False)
print(sim.shape)

(96957, 70)


## Visão geral

In [3]:
resumo = sim.groupby("semeadura")[["ciclo_dias", "PPf", "PA", "ISNA_floracao", "ISNA_enchimento",
                                   "geada_floracao", "geada_enchimento"]].mean()
resumo.loc[[f"{d:02d}/{m:02d}" for m, d in DATAS_SEMEADURA]].round(2)

,ciclo_dias,PPf,PA,ISNA_floracao,ISNA_enchimento,geada_floracao,geada_enchimento
semeadura,,,,,,,
05/01,115.25,7356.46,3029.87,0.73,0.65,0.00,0.09
15/01,120.81,7241.96,2539.72,0.68,0.62,0.00,0.24
25/01,127.79,7113.74,2277.11,0.65,0.60,0.00,0.46
05/02,136.70,6976.51,1982.20,0.62,0.60,0.03,0.76
15/02,144.92,6851.35,1856.85,0.58,0.61,0.09,1.02
25/02,152.43,6780.97,1900.31,0.59,0.63,0.22,1.17
05/03,158.11,6739.26,1968.53,0.60,0.65,0.35,1.20
15/03,164.17,6725.94,1981.99,0.62,0.65,0.49,1.12
25/03,168.42,6760.38,2133.46,0.68,0.60,0.60,0.90
